# Objectively Reviewed: data preparation and NLP analysis

This notebook runs the complete data preparation workflow and initial NLP analysis in one place. It
combines [data cleaning](notebooks/data_cleaning.ipynb),
[HTML extraction](notebooks/extract_html_data.ipynb), and
[data/text merging](notebooks/merge_data_and_text.ipynb), in that order.
The three original notebooks remain available and unchanged.

The goal is to pair cleaned album metadata and audio features with the correct
Pitchfork review text, keep an audit trail for data preparation, and test
whether review language predicts Pitchfork scores and Spotify audio features.

| Stage | Input | What happens | Main output |
| --- | --- | --- | --- |
| 1. Clean metadata | `data.csv` | Normalize text, handle missing values, check audio features, and validate records | `outputs/data_clean.csv` |
| 2. Extract reviews | `pitchfork-scrape/raw-html/` | Parse saved HTML, identify reviews, exclude duplicate groups, and build the text dataset | `outputs/review_text.csv` |
| 3. Merge | Both generated CSVs and the extraction exclusion report | Match artist, album, and date; require author and score agreement | `outputs/full_review.csv` |
| 4. NLP baseline | `outputs/full_review.csv` | Fit TF-IDF/Ridge models and compare with a training-mean baseline | Held-out metrics and selected word coefficients |

## How to run

1. Open this notebook in VS Code or Jupyter and select a Python 3 kernel with
   **NumPy**, **pandas**, and **scikit-learn** installed. The repository's
   `requirements.txt` lists the dependencies; extraction and merging use the
   standard library. The reusable NLP model implementation is in
   `scripts/preliminary_results.py`.
2. Review the paths and cleaning settings in **0. Setup and configuration**.
   The default inputs are the repository's existing CSV and saved HTML files;
   no download or API credentials are required.
3. Restart the kernel and choose **Run All**. Run from top to bottom so each
   stage can use the files and helpers created earlier. HTML extraction over
   the complete collection can take several minutes.
4. The final section reads the generated `outputs/full_review.csv`, fits the
   NLP baselines, and reports held-out results. Review the audit reports under
   `outputs/` before interpreting them.

Paths are anchored to the detected repository root, so this notebook works
with a kernel started in the root, `notebooks/`, or a deeper repository
directory. If the kernel starts elsewhere, set `REPO_ROOT_OVERRIDE` below.
Changing `OUTPUTS_DIR` in the configuration redirects every stage together.

Runs regenerate the CSVs and reports. Before replacing extracted text files,
the extraction stage archives the previous text directory under
`outputs/previous_text_reviews_*/text_reviews/`. The source CSV, saved HTML,
and original notebooks are preserved. After changing settings, restart the
kernel and rerun the full workflow.


## 0. Setup and configuration

Import the dependencies once, then define the paths used by all three stages.
Repository discovery searches the current working directory and its parents
for `data.csv` and `notebooks/`; it does not depend on the notebook being inside
`notebooks/`. No working-directory change is needed.


In [1]:
from collections import Counter, defaultdict
import csv
from datetime import date
from decimal import Decimal, InvalidOperation
import hashlib
from html.parser import HTMLParser
import json
from pathlib import Path
from pprint import pprint
import re
import sys
import tempfile
import unicodedata
from urllib.parse import urlsplit, urlunsplit

from IPython.display import display
import numpy as np
import pandas as pd


In [2]:
# Usually leave this as None. Set an absolute repository path if the kernel
# starts outside this repository, for example Path("/path/to/objectively-reviewed").
REPO_ROOT_OVERRIDE = None

if REPO_ROOT_OVERRIDE is None:
    working_dir = Path.cwd().resolve()
    REPO_ROOT = next(
        (path for path in (working_dir, *working_dir.parents)
         if (path / "data.csv").is_file() and (path / "notebooks").is_dir()),
        None,
    )
else:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository. Start the kernel inside the repository "
        "or set REPO_ROOT_OVERRIDE to its absolute path."
    )

# Edit paths here; all stages share these settings.
RAW_DATA_CSV = REPO_ROOT / "data.csv"
HTML_INPUTS = [REPO_ROOT / "pitchfork-scrape" / "raw-html"]
OUTPUTS_DIR = REPO_ROOT / "outputs"

CLEANING_OUTPUT_DIR = OUTPUTS_DIR / "data_cleaning"
DATA_CLEAN_CSV = OUTPUTS_DIR / "data_clean.csv"
EXTRACTION_DIR = OUTPUTS_DIR / "extracting_text"
TEXT_OUTPUT_DIR = OUTPUTS_DIR / "text_reviews"
REVIEW_TEXT_CSV = OUTPUTS_DIR / "review_text.csv"
DISCARDED_HTML_CSV = EXTRACTION_DIR / "discarded_html.csv"
FULL_REVIEW_CSV = OUTPUTS_DIR / "full_review.csv"

if not RAW_DATA_CSV.is_file():
    raise FileNotFoundError(f"Source CSV does not exist: {RAW_DATA_CSV}")
if not HTML_INPUTS or any(not Path(path).exists() for path in HTML_INPUTS):
    raise FileNotFoundError(f"Check the HTML input paths: {HTML_INPUTS}")

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
CLEANING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Source CSV:", RAW_DATA_CSV)
print("HTML inputs:", HTML_INPUTS)
print("Output directory:", OUTPUTS_DIR)


Repository: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424
Source CSV: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/data.csv
HTML inputs: [PosixPath('/Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/pitchfork-scrape/raw-html')]
Output directory: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs


### Cleaning choices and audit log

These are the defaults from the original cleaning notebook. In particular,
`SUSPECT_FEATURES_ACTION = "drop"` removes every row whose complete audio
feature vector is shared with a different artist; use `"flag"` to retain those
values or `"nan"` to retain the rows with blank audio features. The categorical
musical `key` column is dropped because its album averages are not meaningful.

Missing genres and labels remain missing by default. Release/review conflicts
are flagged, and a review more than two years after release is marked as a
reissue or retrospective. Each cleaning action records its affected count.


In [3]:
# ---- Settings ----------------------------------------------------------------
# Strings that really mean "missing" in text columns (compared case-insensitively)
MISSING_PLACEHOLDERS = ["none", "n/a", "na", "null", "nan", "-", "--", "?", "unknown", ""]

# Replace curly quotes/apostrophes with straight ones in names and titles.
# Makes joins with other sources (e.g. full review text) far more reliable.
NORMALIZE_QUOTES = True

# Drop rows where a key identifying field (artist or album) is missing
DROP_MISSING_KEY_FIELDS = True

# The 'key' column is an album average of a categorical value (0 = C, 1 = C#, ...),
# so it has no musical meaning. True drops it.
DROP_KEY_COLUMN = True

# What to do with rows whose audio features are identical to a different artist's
# album (likely a bad title-only match to Spotify):
#   "flag" -> keep values, add features_suspect = True
#   "nan"  -> flag AND blank the audio features
#   "drop" -> remove the rows entirely
SUSPECT_FEATURES_ACTION = "drop"

# Years between release and review after which a review is treated as a reissue/retrospective
REISSUE_GAP_YEARS = 2

# Fill missing genre / record label with "Unknown" (False keeps them as blank/NaN)
FILL_UNKNOWN_CATEGORIES = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---- Cleaning log ------------------------------------------------------------
cleaning_log = []

def log(step, detail, affected):
    """Record one cleaning action and print it."""
    cleaning_log.append({"step": step, "detail": detail, "affected": int(affected)})
    print(f"[{step}] {detail}: {int(affected):,}")


## 1. Clean the metadata and audio features

This stage starts from `data.csv` on every full run and produces validated
metadata for the later join. `source_row` preserves the zero-based source
record position, while `review_id` identifies rows within this cleaned export.
Neither ID is the URL-derived `source_id` created during HTML extraction.


### 1.1 Read and verify the source file

Read the bytes, try UTF-8, and fall back to Latin-1 if decoding fails. Report
the encoding, byte-order mark, physical line count, and parsed dimensions.
Work on a copy and attach `source_row` before any records are removed.


In [4]:
with open(RAW_DATA_CSV, "rb") as f:
    raw_bytes = f.read()

# Encoding check: fails loudly if the file is not valid UTF-8
try:
    raw_text = raw_bytes.decode("utf-8")
    encoding = "utf-8"
except UnicodeDecodeError:
    raw_text = raw_bytes.decode("latin-1")
    encoding = "latin-1"
print("Encoding:", encoding)
print("Has BOM:", raw_text.startswith("\ufeff"))

line_count = raw_text.count("\n") + (0 if raw_text.endswith("\n") else 1)

alb_data_raw = pd.read_csv(RAW_DATA_CSV, encoding=encoding)
print(f"Lines in file: {line_count:,}  |  Rows parsed: {len(alb_data_raw):,}  |  Columns: {alb_data_raw.shape[1]}")

if line_count - 1 == len(alb_data_raw):
    print("OK: one row per line, no embedded line breaks.")
else:
    print("WARNING: row count differs from line count. Inspect quoted fields with line breaks.")

alb_data = alb_data_raw.copy()
alb_data.insert(0, "source_row", np.arange(len(alb_data)))   # keeps a link back to the raw file
ROWS_AT_START = len(alb_data)


Encoding: utf-8
Has BOM: False
Lines in file: 18,404  |  Rows parsed: 18,403  |  Columns: 18
OK: one row per line, no embedded line breaks.


### 1.2 Profile the raw data

Preview the records, inspect data types, missingness, and distinct values,
and summarize numeric distributions. Define the text and audio-feature
columns used by the checks below; the unit-interval features should lie in
`[0, 1]`.


In [5]:
print("Shape:", alb_data.shape)
display(alb_data.head())


Shape: (18403, 19)


,source_row,artist,album,reviewauthor,score,releaseyear,reviewdate,recordlabel,genre,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0,Studio 1,Studio 1,Andy Battaglia,8.5,2009.0,February 18 2009,Studio,Electronic,0.511917,0.499667,5.250000,-5.626583,0.031983,0.724917,0.024493,0.165367,0.555083,101.395167
1,1,John Fahey,The Great Santa Barbara Oil Slick,Mark Richardson,8.2,2005.0,February 13 2005,Water,Folk/Country,0.369765,0.325412,4.470588,-19.153824,0.148624,0.647053,0.559133,0.527782,0.179465,107.622647
2,2,Reigning Sound,Too Much Guitar,Stephen M. Deusner,8.3,2004.0,August 19 2004,In the Red,Electronic,0.253943,0.912857,4.428571,-1.089500,0.055500,0.000253,0.751214,0.199071,0.552786,133.895500
3,3,The Red Thread,After the Last,Chris Dahlen,7.3,2003.0,July 17 2003,Badman,Rock,0.425400,0.433474,5.700000,-12.871000,0.028260,0.310325,0.224137,0.125150,0.451400,104.354200
4,4,Mac Miller,Swimming,Evan Rytlewski,7.5,2018.0,August 3 2018,Warner Bros.,Rap,0.624846,0.438154,4.153846,-9.456077,0.170246,0.652462,0.012819,0.121131,0.281138,122.121308


In [6]:
profile = pd.DataFrame({
    "dtype": alb_data.dtypes.astype(str),
    "missing": alb_data.isna().sum(),
    "missing_%": (alb_data.isna().mean() * 100).round(2),
    "unique": alb_data.nunique(),
})
profile


,dtype,missing,missing_%,unique
source_row,int64,0,0.00,18403
artist,object,1,0.01,9049
album,object,0,0.00,17714
reviewauthor,object,0,0.00,564
score,float64,0,0.00,100
releaseyear,float64,325,1.77,61
reviewdate,object,0,0.00,4957
recordlabel,object,26,0.14,3286
genre,object,1289,7.00,10
danceability,float64,0,0.00,12407


In [7]:
alb_data.describe().T


,count,mean,std,min,25%,50%,75%,max
source_row,18403.0,9201.000000,5312.632838,0.00000,4600.500000,9201.000000,13801.500000,18402.000
score,18403.0,7.033255,1.251850,0.00000,6.500000,7.300000,7.800000,10.000
releaseyear,18078.0,2009.315411,7.708710,1957.00000,2005.000000,2010.000000,2015.000000,2019.000
danceability,18403.0,0.510635,0.159833,-1.00000,0.405784,0.510417,0.622000,0.974
energy,18403.0,0.600806,0.204257,-1.00000,0.471787,0.624364,0.750586,0.999
key,18403.0,5.215836,1.619708,-1.00000,4.333333,5.230769,6.100000,11.000
loudness,18403.0,-9.318390,4.264038,-51.72875,-11.248350,-8.492333,-6.416397,4.078
speechiness,18403.0,0.091125,0.092428,-1.00000,0.040704,0.056808,0.100029,0.958
acousticness,18403.0,0.302895,0.263730,-1.00000,0.089842,0.229806,0.463416,0.996
instrumentalness,18403.0,0.278616,0.300740,-1.00000,0.010060,0.153433,0.506816,0.982


In [8]:
TEXT_COLS = ["artist", "album", "reviewauthor", "recordlabel", "genre"]
FEATURE_COLS = ["danceability", "energy", "key", "loudness", "speechiness",
                "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
UNIT_INTERVAL_COLS = ["danceability", "energy", "speechiness", "acousticness",
                      "instrumentalness", "liveness", "valence"]   # should lie in [0, 1]


### 1.3 Remove exact duplicate records

Compare every original data column, excluding the tracking column
`source_row`. Display duplicate groups and retain the first identical record.
Later stages handle repeated review identities separately because rows can
share a review key while differing in another field, such as record label.


In [9]:
data_cols = [c for c in alb_data.columns if c != "source_row"]
dup_mask = alb_data.duplicated(subset=data_cols, keep="first")
display(alb_data[alb_data.duplicated(subset=data_cols, keep=False)][["source_row", "artist", "album", "reviewauthor", "reviewdate"]])
alb_data = alb_data[~dup_mask].copy()
log("Duplicates", "Exact duplicate rows removed", dup_mask.sum())


,source_row,artist,album,reviewauthor,reviewdate
8012,8012,Fiona Apple,When the Pawn...,Judy Berman,March 24 2019
8013,8013,Fiona Apple,When the Pawn...,Judy Berman,March 24 2019
12874,12874,Stick Men with Ray Guns,Some People Deserve to Suffer,Andy Beta,March 5 2003
12875,12875,Stick Men with Ray Guns,Some People Deserve to Suffer,Andy Beta,March 5 2003


[Duplicates] Exact duplicate rows removed: 2


### 1.4 Normalize text formatting

Normalize Unicode, optionally straighten quotes, replace non-breaking
spaces, and collapse repeated whitespace in names, titles, authors, labels,
and genres. Log how many values change in each column. The cleaning quote
map has its own name so it remains independent of the later matching rules.


In [10]:
CLEANING_QUOTE_MAP = str.maketrans({"\u201c": '"', "\u201d": '"', "\u2018": "'", "\u2019": "'",
                           "\u00a0": " "})   # also replaces non-breaking spaces

def normalize_text(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFC", str(s))
    if NORMALIZE_QUOTES:
        s = s.translate(CLEANING_QUOTE_MAP)
    return " ".join(s.split())    # trims and collapses all runs of whitespace

for col in TEXT_COLS:
    before_unique = alb_data[col].nunique()
    new = alb_data[col].map(normalize_text)
    changed = (new.fillna("<NA>") != alb_data[col].fillna("<NA>")).sum()
    alb_data[col] = new
    log("Text normalisation", f"{col}: values changed ({before_unique:,} -> {alb_data[col].nunique():,} unique)", changed)


[Text normalisation] artist: values changed (9,049 -> 9,047 unique): 140
[Text normalisation] album: values changed (17,714 -> 17,711 unique): 150
[Text normalisation] reviewauthor: values changed (564 -> 559 unique): 83
[Text normalisation] recordlabel: values changed (3,286 -> 3,283 unique): 41


[Text normalisation] genre: values changed (10 -> 10 unique): 0


### 1.5 Convert placeholder strings to missing values

Treat case-insensitive entries such as `unknown`, `n/a`, and `--` as missing
according to `MISSING_PLACEHOLDERS`. Report their frequency and replace them
with `NaN`, allowing missing-value checks to detect them consistently.


In [11]:
placeholders = {p.lower() for p in MISSING_PLACEHOLDERS}

for col in TEXT_COLS:
    mask = alb_data[col].notna() & alb_data[col].str.lower().isin(placeholders)
    if mask.any():
        print(f"{col}: placeholder values found -> {alb_data.loc[mask, col].value_counts().to_dict()}")
    alb_data.loc[mask, col] = np.nan
    log("Disguised missing", f"{col}: placeholder strings -> NaN", mask.sum())


[Disguised missing] artist: placeholder strings -> NaN: 0
[Disguised missing] album: placeholder strings -> NaN: 0
[Disguised missing] reviewauthor: placeholder strings -> NaN: 0
recordlabel: placeholder values found -> {'none': 607, 'Unknown': 7}
[Disguised missing] recordlabel: placeholder strings -> NaN: 614
genre: placeholder values found -> {'none': 11}
[Disguised missing] genre: placeholder strings -> NaN: 11


### 1.6 Unify capitalization variants

For artists, authors, labels, and genres, group case-insensitive variants
and use the most frequent spelling in each group. Album titles keep their
capitalization; this step does not guess corrections for distinct names.


In [12]:
def unify_case_variants(series):
    s = series.dropna()
    counts = s.value_counts()
    canonical = (counts.rename_axis("value").reset_index()
                       .assign(key=lambda d: d["value"].str.casefold())
                       .sort_values(["key", "count"], ascending=[True, False])
                       .drop_duplicates("key")
                       .set_index("key")["value"])
    groups = s.groupby(s.str.casefold()).agg(lambda x: sorted(set(x)))
    variants = groups[groups.str.len() > 1]
    return series.map(lambda v: canonical[v.casefold()] if pd.notna(v) else v), variants

for col in ["artist", "reviewauthor", "recordlabel", "genre"]:
    new, variants = unify_case_variants(alb_data[col])
    changed = (new.fillna("<NA>") != alb_data[col].fillna("<NA>")).sum()
    if len(variants):
        print(f"\n{col}: {len(variants)} case-variant groups, e.g. {variants.head(5).tolist()}")
    alb_data[col] = new
    log("Case variants", f"{col}: values unified to most common spelling", changed)



artist: 3 case-variant groups, e.g. [['Billy Woods', 'billy woods'], ['The Comet Is Coming', 'The Comet is Coming'], ['WIVES', 'Wives']]


[Case variants] artist: values unified to most common spelling: 3
[Case variants] reviewauthor: values unified to most common spelling: 0



recordlabel: 2 case-variant groups, e.g. [['GOOD', 'Good'], ['ITALIANS DO IT BETTER', 'Italians Do It Better']]
[Case variants] recordlabel: values unified to most common spelling: 3
[Case variants] genre: values unified to most common spelling: 0


### 1.7 Convert dates and release years

Parse review dates in the source's month/day/year format and report failures.
Verify that nonmissing release years are integers, then store them using
pandas' nullable integer type. Optionally fill missing genres and labels with
`Unknown` if the corresponding setting is enabled.


In [13]:
parsed = pd.to_datetime(alb_data["reviewdate"], format="%B %d %Y", errors="coerce")
unparsed = parsed.isna() & alb_data["reviewdate"].notna()
if unparsed.any():
    print("Dates that could not be parsed:", alb_data.loc[unparsed, "reviewdate"].head().tolist())
alb_data["reviewdate"] = parsed
log("Types", "reviewdate parsed to datetime (failures)", unparsed.sum())

non_integer_years = (alb_data["releaseyear"].dropna() % 1 != 0).sum()
assert non_integer_years == 0, "releaseyear has non-integer values"
alb_data["releaseyear"] = alb_data["releaseyear"].astype("Int64")
log("Types", "releaseyear converted to nullable integer (missing values)", alb_data["releaseyear"].isna().sum())

for col in ["genre", "recordlabel"]:
    if FILL_UNKNOWN_CATEGORIES:
        n = alb_data[col].isna().sum()
        alb_data[col] = alb_data[col].fillna("Unknown")
        log("Types", f"{col}: missing filled with 'Unknown'", n)

print(alb_data[["reviewdate", "releaseyear"]].dtypes)


[Types] reviewdate parsed to datetime (failures): 0
[Types] releaseyear converted to nullable integer (missing values): 325
reviewdate     datetime64[ns]
releaseyear             Int64
dtype: object


### 1.8 Handle missing artist or album

Display rows without the fields needed to identify an album. With the
default settings, remove those rows and record the count; the configuration
can instead keep them for inspection.


In [14]:
key_missing = alb_data["artist"].isna() | alb_data["album"].isna()
display(alb_data.loc[key_missing, ["source_row", "artist", "album", "reviewauthor", "reviewdate"]])
if DROP_MISSING_KEY_FIELDS:
    alb_data = alb_data[~key_missing].copy()
    log("Missing keys", "Rows dropped (missing artist or album)", key_missing.sum())
else:
    log("Missing keys", "Rows with missing artist or album (kept)", key_missing.sum())


,source_row,artist,album,reviewauthor,reviewdate
4720,4720,NaN,Cellar EP,Kevin Lozano,2016-02-29


[Missing keys] Rows dropped (missing artist or album): 1


### 1.9 Validate audio-feature values

Replace the `-1` missing-value sentinel with `NaN` and blank invalid values
in features restricted to `[0, 1]`. Flag positive loudness and tempo outside
50–250 BPM for inspection. Drop the averaged `key` column when configured,
and update the feature list used by subsequent checks.


In [15]:
sentinel_any = (alb_data[FEATURE_COLS] == -1).any(axis=1)
sentinel_all = (alb_data[FEATURE_COLS] == -1).all(axis=1)
print(f"Rows with -1 in all features: {sentinel_all.sum()}, in only some: {(sentinel_any & ~sentinel_all).sum()}")
alb_data[FEATURE_COLS] = alb_data[FEATURE_COLS].mask(alb_data[FEATURE_COLS] == -1)
log("Audio features", "Rows with -1 sentinel values set to NaN", sentinel_any.sum())

# Out-of-range values: a negative album average usually means one track carried the -1
# sentinel and was averaged in, so the value is contaminated and can't be trusted.
oor = (alb_data[UNIT_INTERVAL_COLS] < 0) | (alb_data[UNIT_INTERVAL_COLS] > 1)
print("\nValues outside [0, 1] after sentinel removal:\n", oor.sum().to_string())
if oor.any().any():
    display(alb_data.loc[oor.any(axis=1), ["artist", "album"] + UNIT_INTERVAL_COLS])
alb_data[UNIT_INTERVAL_COLS] = alb_data[UNIT_INTERVAL_COLS].mask(oor)
log("Audio features", "Out-of-range [0, 1] feature values set to NaN", oor.sum().sum())

alb_data["loudness_positive"] = alb_data["loudness"] > 0
alb_data["tempo_unusual"] = (alb_data["tempo"] < 50) | (alb_data["tempo"] > 250)
log("Audio features", "Rows flagged with positive loudness", alb_data["loudness_positive"].sum())
log("Audio features", "Rows flagged with unusual tempo (<50 or >250 BPM)", alb_data["tempo_unusual"].sum())

if DROP_KEY_COLUMN:
    non_int = (alb_data["key"].dropna() % 1 != 0).sum()
    print(f"\n'key' is non-integer (averaged) in {non_int:,} rows -> dropping column")
    alb_data = alb_data.drop(columns="key")
    FEATURE_COLS = [c for c in FEATURE_COLS if c != "key"]
    log("Audio features", "'key' column dropped (rows with averaged values)", non_int)


Rows with -1 in all features: 8, in only some: 0
[Audio features] Rows with -1 sentinel values set to NaN: 8

Values outside [0, 1] after sentinel removal:
 danceability        0
energy              0
speechiness         1
acousticness        0
instrumentalness    1
liveness            0
valence             0


,artist,album,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence
12570,Rahim,Ideal Lives,0.518364,0.531909,-0.058236,0.012756,-0.085783,0.075236,0.462364


[Audio features] Out-of-range [0, 1] feature values set to NaN: 2
[Audio features] Rows flagged with positive loudness: 9
[Audio features] Rows flagged with unusual tempo (<50 or >250 BPM): 5

'key' is non-integer (averaged) in 15,089 rows -> dropping column
[Audio features] 'key' column dropped (rows with averaged values): 15,089


### 1.10 Handle suspected wrong Spotify matches

Find complete feature vectors that are identical across different artists
after rounding to six decimal places. These can indicate a title-only match
to the wrong Spotify album, and the file does not identify which row is
correct. Flag every member of each group, then apply
`SUSPECT_FEATURES_ACTION` (`drop` by default).


In [16]:
has_features = alb_data[FEATURE_COLS].notna().all(axis=1)
feat_key = alb_data.loc[has_features, FEATURE_COLS].round(6).astype(str).agg("|".join, axis=1)
artists_per_vector = alb_data.loc[has_features].groupby(feat_key)["artist"].transform("nunique")

alb_data["features_suspect"] = False
alb_data.loc[artists_per_vector.index, "features_suspect"] = artists_per_vector > 1

examples = (alb_data[alb_data["features_suspect"]]
            .assign(_k=feat_key)
            .sort_values(["_k", "artist"])
            [["artist", "album", "energy", "tempo"]]
            .head(10))
display(examples)
log("Suspect features", "Rows sharing identical features with a different artist", alb_data["features_suspect"].sum())

if SUSPECT_FEATURES_ACTION == "nan":
    alb_data.loc[alb_data["features_suspect"], FEATURE_COLS] = np.nan
    log("Suspect features", "Suspect rows: audio features set to NaN", alb_data["features_suspect"].sum())
elif SUSPECT_FEATURES_ACTION == "drop":
    n = alb_data["features_suspect"].sum()
    alb_data = alb_data[~alb_data["features_suspect"]].copy()
    log("Suspect features", "Suspect rows dropped", n)


,artist,album,energy,tempo
4397,Muse,Drones,0.027630,81.425000
4398,Nico Muhly,Drones,0.027630,81.425000
12635,Nosaj Thing,Drift,0.079600,153.779000
12636,The Men,Drift,0.079600,153.779000
1506,Kath Bloom,Thin Thin Line,0.111600,82.949818
1505,Trans Am,The Red Line,0.111600,82.949818
6464,Hepcat,Out of Nowhere,0.637000,200.085000
6463,Jimi Tenor,Out of Nowhere,0.637000,200.085000
7870,Carmen Villain,Sleeper,0.123675,115.411750
7871,Ty Segall,Sleeper,0.123675,115.411750


[Suspect features] Rows sharing identical features with a different artist: 1,589
[Suspect features] Suspect rows dropped: 1,589


### 1.11 Check release year against review date

Calculate `years_since_release`. Mark gaps greater than
`REISSUE_GAP_YEARS` with `is_reissue`, and negative gaps with
`release_after_review`. Keep these rows with their flags rather than
changing their dates or scores.


In [17]:
review_year = alb_data["reviewdate"].dt.year
gap = (review_year - alb_data["releaseyear"]).astype("Int64")
alb_data["years_since_release"] = gap

alb_data["is_reissue"] = (gap > REISSUE_GAP_YEARS).fillna(False).astype(bool)
alb_data["release_after_review"] = (gap < 0).fillna(False).astype(bool)

log("Cross-field", f"Reviews flagged as reissue (> {REISSUE_GAP_YEARS} years after release)", alb_data["is_reissue"].sum())
log("Cross-field", "Rows where release year is after review year (flagged)", alb_data["release_after_review"].sum())
display(alb_data.loc[alb_data["release_after_review"], ["artist", "album", "releaseyear", "reviewdate"]].head())


[Cross-field] Reviews flagged as reissue (> 2 years after release): 786
[Cross-field] Rows where release year is after review year (flagged): 24


,artist,album,releaseyear,reviewdate
124,The Avalanches,Since I Left You,2001,1999-12-31
532,Love Is All,Nine Times That Same Song,2006,2005-12-05
1215,Abilene,Abilene,2001,2000-12-31
3685,Cave In,Jupiter,2001,2000-08-08
4647,Alpha,The Impossible Thrill,2001,2000-12-31


### 1.12 Identify repeat reviews across dates

Group albums by case-insensitive artist and album, then order each group by
review date. Add a review count and sequence number. Reviews on different
dates remain eligible, allowing an original review and a later retrospective
to be distinguished during merging.


In [18]:
album_key = alb_data["artist"].str.casefold() + " || " + alb_data["album"].str.casefold()
alb_data = alb_data.assign(_album_key=album_key).sort_values(["_album_key", "reviewdate"])
alb_data["album_review_count"] = alb_data.groupby("_album_key")["_album_key"].transform("size")
alb_data["album_review_number"] = alb_data.groupby("_album_key").cumcount() + 1
alb_data = alb_data.drop(columns="_album_key").sort_values("source_row")

repeats = alb_data[alb_data["album_review_count"] > 1]
log("Repeat reviews", "Rows belonging to albums reviewed more than once (kept)", len(repeats))
display(repeats[["artist", "album", "reviewauthor", "score", "reviewdate", "is_reissue"]].head(8))


[Repeat reviews] Rows belonging to albums reviewed more than once (kept): 88


,artist,album,reviewauthor,score,reviewdate,is_reissue
12,American Football,American Football,Ian Cohen,8.6,2014-05-21,True
13,American Football,American Football,Ian Cohen,7.7,2016-10-26,False
99,Crystal Castles,Crystal Castles,Ian Cohen,7.8,2008-03-17,False
100,Crystal Castles,Crystal Castles,Ian Cohen,8.5,2010-04-29,False
321,Ty Segall,Ty Segall,Stuart Berman,8.0,2017-01-25,False
322,Ty Segall,Ty Segall,Stephen M. Deusner,6.7,2009-07-17,False
399,Atmosphere,God Loves Ugly,Nate Patrin,6.6,2009-02-20,True
400,Atmosphere,God Loves Ugly,Sam Chennault,5.7,2002-07-04,False


### 1.13 Assign review IDs and arrange columns

Restore source order, assign sequential `review_id` values, and place
identity fields, audio features, helper columns, and flags in a predictable
order. `source_row` remains available for tracing a row to `data.csv`.


In [19]:
alb_data = alb_data.sort_values("source_row").reset_index(drop=True)
alb_data.insert(0, "review_id", np.arange(1, len(alb_data) + 1))

flag_cols = ["is_reissue", "release_after_review", "features_suspect",
             "loudness_positive", "tempo_unusual"]
helper_cols = ["years_since_release", "album_review_count", "album_review_number"]
core_cols = ["review_id", "source_row", "artist", "album", "reviewauthor", "score",
             "releaseyear", "reviewdate", "recordlabel", "genre"]
alb_data = alb_data[core_cols + FEATURE_COLS + helper_cols + flag_cols]
alb_data.head()


,review_id,source_row,artist,album,reviewauthor,score,releaseyear,reviewdate,recordlabel,genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,years_since_release,album_review_count,album_review_number,is_reissue,release_after_review,features_suspect,loudness_positive,tempo_unusual
0,1,0,Studio 1,Studio 1,Andy Battaglia,8.5,2009,2009-02-18,Studio,Electronic,0.511917,0.499667,-5.626583,0.031983,0.724917,0.024493,0.165367,0.555083,101.395167,0,1,1,False,False,False,False,False
1,2,1,John Fahey,The Great Santa Barbara Oil Slick,Mark Richardson,8.2,2005,2005-02-13,Water,Folk/Country,0.369765,0.325412,-19.153824,0.148624,0.647053,0.559133,0.527782,0.179465,107.622647,0,1,1,False,False,False,False,False
2,3,2,Reigning Sound,Too Much Guitar,Stephen M. Deusner,8.3,2004,2004-08-19,In the Red,Electronic,0.253943,0.912857,-1.089500,0.055500,0.000253,0.751214,0.199071,0.552786,133.895500,0,1,1,False,False,False,False,False
3,4,3,The Red Thread,After the Last,Chris Dahlen,7.3,2003,2003-07-17,Badman,Rock,0.425400,0.433474,-12.871000,0.028260,0.310325,0.224137,0.125150,0.451400,104.354200,0,1,1,False,False,False,False,False
4,5,6,Jessica Lea Mayfield,With Blasphemy So Heartfelt,Stephen M. Deusner,8.2,2008,2008-11-11,Polymer,Rock,0.447583,0.399083,-8.402000,0.032267,0.694750,0.004033,0.123142,0.237933,111.549917,0,1,1,False,False,False,False,False


### 1.14 Validate the cleaned dataset

Stop execution if IDs are duplicated, exact duplicates remain, scores or
feature values violate their allowed ranges, dates fail to parse, or text
still contains whitespace and missing-placeholder problems. Show the final
data types and missingness after the assertions pass.


In [20]:
assert alb_data["review_id"].is_unique, "review_id is not unique"
assert not alb_data.drop(columns=["review_id", "source_row"]).duplicated().any(), "Exact duplicates remain"
assert alb_data["score"].between(0, 10).all(), "Score outside 0-10"
assert alb_data["reviewdate"].notna().all(), "Unparsed review dates"
assert alb_data["releaseyear"].dropna().between(1900, alb_data["reviewdate"].dt.year.max() + 1).all(), "Implausible release year"
assert (alb_data[FEATURE_COLS] != -1).all().all(), "Sentinel -1 values remain"
assert ((alb_data[UNIT_INTERVAL_COLS] >= 0) & (alb_data[UNIT_INTERVAL_COLS] <= 1) | alb_data[UNIT_INTERVAL_COLS].isna()).all().all(), "Features outside [0, 1]"
for col in ["artist", "album", "reviewauthor", "recordlabel", "genre"]:
    s = alb_data[col].dropna()
    assert (s == s.str.strip()).all(), f"{col}: leading/trailing whitespace remains"
    assert not s.str.contains("  ").any(), f"{col}: double spaces remain"
    assert not s.str.lower().isin({p.lower() for p in MISSING_PLACEHOLDERS}).any(), f"{col}: placeholder strings remain"
if DROP_MISSING_KEY_FIELDS:
    assert alb_data[["artist", "album"]].notna().all().all(), "Missing artist/album remain"
print("All validation checks passed.")


All validation checks passed.

In [21]:
pd.DataFrame({
    "dtype": alb_data.dtypes.astype(str),
    "missing": alb_data.isna().sum(),
    "missing_%": (alb_data.isna().mean() * 100).round(2),
})


,dtype,missing,missing_%
review_id,int64,0,0.00
source_row,int64,0,0.00
artist,object,0,0.00
album,object,0,0.00
reviewauthor,object,0,0.00
score,float64,0,0.00
releaseyear,Int64,301,1.79
reviewdate,datetime64[ns],0,0.00
recordlabel,object,575,3.42
genre,object,1192,7.09


### 1.15 Export the cleaned metadata and log

Write `outputs/data_cleaning/log_df.csv` with one record per cleaning action.
Save `outputs/data_clean.csv` as UTF-8 with ISO-formatted review dates, then
read it back to verify the row count and ID uniqueness. The next stage builds
the review-text dataset from the saved HTML.


In [22]:
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(CLEANING_OUTPUT_DIR / "log_df.csv", index=False, encoding="utf-8")
print(f"Rows: {ROWS_AT_START:,} -> {len(alb_data):,}  ({ROWS_AT_START - len(alb_data):,} removed)")
log_df


Rows: 18,403 -> 16,811  (1,592 removed)


,step,detail,affected
0,Duplicates,Exact duplicate rows removed,2
1,Text normalisation,"artist: values changed (9,049 -> 9,047 unique)",140
2,Text normalisation,"album: values changed (17,714 -> 17,711 unique)",150
3,Text normalisation,reviewauthor: values changed (564 -> 559 unique),83
4,Text normalisation,"recordlabel: values changed (3,286 -> 3,283 un...",41
5,Text normalisation,genre: values changed (10 -> 10 unique),0
6,Disguised missing,artist: placeholder strings -> NaN,0
7,Disguised missing,album: placeholder strings -> NaN,0
8,Disguised missing,reviewauthor: placeholder strings -> NaN,0
9,Disguised missing,recordlabel: placeholder strings -> NaN,614


In [23]:
alb_data.to_csv(DATA_CLEAN_CSV, index=False, encoding="utf-8", date_format="%Y-%m-%d")

check = pd.read_csv(DATA_CLEAN_CSV, parse_dates=["reviewdate"])
assert len(check) == len(alb_data), "Row count changed on round-trip"
assert check["review_id"].is_unique
print(f"Saved {len(check):,} rows x {check.shape[1]} columns to {DATA_CLEAN_CSV}")


Saved 16,811 rows x 27 columns to /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/data_clean.csv


## 2. Extract review text from the saved HTML

This stage contains the HTML parsing and text-to-CSV logic from
`extract_html_data.ipynb`. It reads the paths in `HTML_INPUTS` recursively,
uses a canonical review URL to identify each review, and keeps names and
punctuation as they appear in the source headings.

Each retained review gets `outputs/text_reviews/<source_id>.txt`. The
manifest connects that filename to its source HTML and identity fields.
Reviews missing a description are reported and their text files are kept,
but they are excluded from `review_text.csv`. Duplicate review groups are
discarded in full and reported for the merge stage.


### 2.1 Define shared schemas and matching helpers

The metadata CSV contains only `source_id`, `artist`, `album`, `reviewdate`,
`reviewauthor`, `score`, `description`, and `review`. The separate manifest
also records the HTML and text filenames.

Matching normalization handles Unicode, curly quotes, whitespace, and case.
It does not change the names saved in the CSV or remove punctuation.
The normalization, identity, and CSV helpers below are shared with the
merge stage and defined only once. Canonical URLs use HTTPS and a consistent
host and trailing slash, without tracking parameters or fragments. The first
32 hexadecimal characters of the URL's SHA-256 hash form `source_id`.


In [24]:
IDENTITY_FIELDS = [
    "source_id", "artist", "album", "reviewdate", "reviewauthor", "score"
]


REVIEW_FIELDS = IDENTITY_FIELDS + ["description", "review"]


MAP_FIELDS = ["source_id", "html_file_name", "text_file_name"] + IDENTITY_FIELDS[1:]


In [25]:
QUOTE_MAP = str.maketrans({
    "‘": "'", "’": "'", "‛": "'", "′": "'", "“": '"', "”": '"', "″": '"',
})


def normalize_name(value):
    """Normalize Unicode, quotes, whitespace, and case without losing punctuation."""
    return " ".join(
        unicodedata.normalize("NFKC", value or "").translate(QUOTE_MAP).split()
    ).casefold()


def review_date(value):
    return date.fromisoformat((value or "").strip()[:10]).isoformat()


def review_score(value):
    try:
        score = Decimal((value or "").strip())
    except InvalidOperation as exc:
        raise ValueError(f"invalid review score: {value!r}") from exc
    if not score.is_finite() or not 0 <= score <= 10:
        raise ValueError(f"invalid review score: {value!r}")
    return score


def match_key(row):
    artist, album = (normalize_name(row.get(field)) for field in ("artist", "album"))
    if not artist or not album:
        raise ValueError("missing artist or album")
    return artist, album, review_date(row.get("reviewdate"))


def duplicate_keys(rows):
    counts = Counter()
    for row in rows:
        try:
            counts[match_key(row)] += 1
        except ValueError:
            continue
    return {key for key, count in counts.items() if count > 1}


In [26]:
def canonical_url(value):
    parts = urlsplit((value or "").strip())
    if parts.hostname not in {"pitchfork.com", "www.pitchfork.com"}:
        raise ValueError("missing or invalid canonical Pitchfork URL")
    if not parts.path.startswith("/reviews/albums/"):
        raise ValueError("canonical URL is not an album review")
    return urlunsplit(("https", "pitchfork.com", parts.path.rstrip("/") + "/", "", ""))


def source_id_for_url(value):
    return hashlib.sha256(canonical_url(value).encode("utf-8")).hexdigest()[:32]


def read_csv(path, required=()):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        fields = reader.fieldnames or []
        missing = set(required) - set(fields)
        if missing:
            raise ValueError(f"{path}: missing columns {sorted(missing)}")
        if len(set(fields)) != len(fields):
            raise ValueError(f"{path}: duplicate column names")
        return fields, list(reader)


def write_csv(path, fields, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


### 2.2 Parse the original HTML

Read the first artist link and first album heading, rather than the combined
HTML title. Publication date and author come from JSON-LD; score comes from the
first score element. The meta description is extracted separately from the
review paragraphs. Invalid or incomplete pages are recorded in the skipped log.


In [27]:
class ReviewParser(HTMLParser):
    """Read visible identity fields, JSON-LD, description, and review paragraphs."""

    VOID_TAGS = {
        "area", "base", "br", "col", "embed", "hr", "img", "input", "link",
        "meta", "param", "source", "track", "wbr",
    }
    IGNORED_TAGS = {"script", "style", "noscript", "iframe", "svg"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.stack = []
        self.values = {}
        self.captures = {}
        self.description = ""
        self.url = ""
        self.og_url = ""
        self.json_blocks = []
        self.json_parts = None
        self.json_depth = None
        self.artist_list_depth = None
        self.paragraphs = []
        self.paragraph_parts = None
        self.review_depth = None

    def capture(self, name):
        if name not in self.values and name not in self.captures:
            self.captures[name] = (len(self.stack), [])

    def finish_paragraph(self):
        if self.paragraph_parts is not None:
            paragraph = " ".join("".join(self.paragraph_parts).split())
            if paragraph:
                self.paragraphs.append(paragraph)
            self.paragraph_parts = None

    def handle_starttag(self, tag, attrs):
        ignored = any(name in self.IGNORED_TAGS for name in self.stack)
        attrs = dict(attrs)
        if not ignored:
            if tag == "meta":
                name = (attrs.get("name") or attrs.get("property") or "").lower()
                if name == "description" and not self.description:
                    self.description = " ".join((attrs.get("content") or "").split())
                elif name == "og:url":
                    self.og_url = attrs.get("content") or ""
            if tag == "link" and "canonical" in (attrs.get("rel") or "").split():
                self.url = attrs.get("href") or ""
        if tag == "br" and not ignored:
            if self.paragraph_parts is not None:
                self.paragraph_parts.append(" ")
            for _, parts in self.captures.values():
                parts.append(" ")
        if tag in self.VOID_TAGS:
            return
        self.stack.append(tag)
        if ignored:
            return
        classes = (attrs.get("class") or "").split()
        if tag == "script" and attrs.get("type") == "application/ld+json":
            self.json_parts = []
            self.json_depth = len(self.stack)
        if tag == "ul" and "artist-links" in classes and "artist" not in self.values:
            self.artist_list_depth = len(self.stack)
        if tag == "li" and self.artist_list_depth is not None:
            self.capture("artist")
        if tag == "h1" and "single-album-tombstone__review-title" in classes:
            self.capture("album")
        if tag == "span" and "score" in classes:
            self.capture("score")
        if tag == "div" and "contents" in classes and self.review_depth is None:
            self.review_depth = len(self.stack)
        if tag == "p" and self.review_depth is not None:
            self.finish_paragraph()
            self.paragraph_parts = []

    def handle_endtag(self, tag):
        if tag not in self.stack:
            return
        index = len(self.stack) - 1 - self.stack[::-1].index(tag)
        ignored = any(name in self.IGNORED_TAGS for name in self.stack)
        for name, (depth, parts) in list(self.captures.items()):
            if index < depth:
                value = " ".join("".join(parts).split())
                # An empty first heading must not select a later album's name.
                self.values[name] = value
                del self.captures[name]
        if self.json_depth is not None and index < self.json_depth:
            self.json_blocks.append("".join(self.json_parts))
            self.json_parts = self.json_depth = None
        if self.artist_list_depth is not None and index < self.artist_list_depth:
            self.artist_list_depth = None
        if tag == "p" and not ignored:
            self.finish_paragraph()
        if self.review_depth is not None and index < self.review_depth:
            self.finish_paragraph()
            self.review_depth = None
        del self.stack[index:]

    def handle_data(self, data):
        if self.json_parts is not None:
            self.json_parts.append(data)
            return
        if any(tag in self.IGNORED_TAGS for tag in self.stack):
            return
        for _, parts in self.captures.values():
            parts.append(data)
        if self.paragraph_parts is not None:
            self.paragraph_parts.append(data)


In [28]:
def json_objects(value):
    """Support plain JSON-LD objects, lists, and @graph documents."""
    if isinstance(value, list):
        for item in value:
            yield from json_objects(item)
    elif isinstance(value, dict):
        yield value
        if "@graph" in value:
            yield from json_objects(value["@graph"])


def extract_review(html):
    parser = ReviewParser()
    parser.feed(html)
    parser.close()
    parser.finish_paragraph()
    if not parser.paragraphs:
        raise ValueError("no review paragraphs found in a div.contents container")
    for field in ("artist", "album", "score"):
        if not parser.values.get(field):
            raise ValueError(f"no nonempty {field} field found in the review heading")
    article = None
    for block in parser.json_blocks:
        try:
            candidates = json_objects(json.loads(block))
            article = next((obj for obj in candidates if obj.get("datePublished")), None)
        except json.JSONDecodeError:
            continue
        if article:
            break
    if not article:
        raise ValueError("no publication metadata found in JSON-LD")
    author = article.get("author") or article.get("creator") or {}
    if isinstance(author, list):
        author = author[0] if len(author) == 1 else {}
    author = author.get("name", "") if isinstance(author, dict) else author
    author = " ".join((author or "").split())
    if not author:
        raise ValueError("no review author found")
    url = canonical_url(parser.url or parser.og_url or article.get("url"))
    return {
        "source_id": source_id_for_url(url),
        "artist": parser.values["artist"],
        "album": parser.values["album"],
        "reviewdate": review_date(article.get("datePublished")),
        "reviewauthor": author,
        "score": str(review_score(parser.values["score"])),
        "description": parser.description,
        "review": "\n\n".join(parser.paragraphs) + "\n",
    }


### 2.3 Find HTML inputs and extract reviews

The complete collection can take a few minutes. Progress is printed every
2,000 files. No existing output is replaced during this parsing step.


In [29]:
files = set()
for source in map(Path, HTML_INPUTS):
    if source.is_dir():
        files.update(
            path.resolve() for path in source.rglob("*")
            if path.is_file() and path.suffix.lower() in {".html", ".htm"}
        )
    elif source.is_file():
        files.add(source.resolve())
    else:
        raise FileNotFoundError(f"Input does not exist: {source}")
if not files:
    raise ValueError("No HTML files found.")
print(f"Found {len(files):,} HTML files.")


Found 21,321 HTML files.


In [30]:
records, skipped = [], []
for index, source in enumerate(sorted(files), 1):
    try:
        record = extract_review(source.read_text(encoding="utf-8-sig"))
        records.append({
            **record,
            "html_file_name": source.name,
            "text_file_name": record["source_id"] + ".txt",
        })
    except (OSError, UnicodeError, ValueError) as exc:
        skipped.append(f"{source}\t{exc}\n")
        print(f"Skipping {source.name}: {exc}", file=sys.stderr)
    if index % 2000 == 0:
        print(f"Parsed {index}/{len(files)} HTML files.", flush=True)
if not records:
    raise ValueError("No reviews could be extracted; previous outputs were preserved.")

print(f"Parsed {len(records):,} reviews; skipped {len(skipped):,} source files.")
for row in records[:3]:
    pprint({field: row[field] for field in IDENTITY_FIELDS})


Skipping 11617-partie-t.html: no review paragraphs found in a div.contents container


Parsed 2000/21321 HTML files.


Parsed 4000/21321 HTML files.


Skipping 1391-being-ridden.html: no review paragraphs found in a div.contents container


Parsed 6000/21321 HTML files.


Skipping 16697-record.html: no nonempty album field found in the review heading


... (6 outputs left out)

Parsed 18000/21321 HTML files.


Skipping 9464-shine-on.html: no review paragraphs found in a div.contents container


Parsed 20000/21321 HTML files.


Parsed 21,316 reviews; skipped 5 source files.
{'album': 'TFCF',
 'artist': 'Liars',
 'reviewauthor': 'Patric Fallon',
 'reviewdate': '2017-08-28',
 'score': '7.5',
 'source_id': 'c7af38b5e299e7cdece3756934c1e947'}
{'album': 'Connected and Respected',
 'artist': 'E-40',
 'reviewauthor': 'Torii MacAdams',
 'reviewdate': '2018-04-14',
 'score': '6.7',
 'source_id': '03b726ca0a9b1a25d770fb7a42cdebfc'}
{'album': '40oz. to Freedom',
 'artist': 'Sublime',
 'reviewauthor': 'Evan Rytlewski',
 'reviewdate': '2018-02-25',
 'score': '5.6',
 'source_id': '22cf35b500b7ee65b2b0f2f9988a748b'}


### 2.4 Exclude duplicate HTML groups and save text files

Discard all copies of a repeated source ID or normalized artist/album/date
key. Stage the new text files before archiving the previous text directory.
Write all four support files to `outputs/extracting_text/`.


In [31]:
def write_outputs(records, skipped, output_dir, reports_dir):
    """Discard entire duplicate groups and replace the text directory together."""
    duplicate_ids = {
        value for value, count in Counter(row["source_id"] for row in records).items()
        if count > 1
    }
    duplicated_keys = duplicate_keys(records)
    kept, discarded = [], []
    for record in records:
        if record["source_id"] in duplicate_ids or match_key(record) in duplicated_keys:
            discarded.append({
                **{field: record[field] for field in MAP_FIELDS},
                "discard_reason": "duplicate_html_review",
            })
        else:
            kept.append(record)

    output_dir.parent.mkdir(parents=True, exist_ok=True)
    backup = None
    with tempfile.TemporaryDirectory(prefix=".review_stage_", dir=output_dir.parent) as tmp:
        stage = Path(tmp) / "text_reviews"
        stage.mkdir()
        for record in kept:
            text = record["review"]
            if record["description"]:
                text = record["description"] + "\n\n" + text
            (stage / record["text_file_name"]).write_text(text, encoding="utf-8")
        if output_dir.exists():
            backup = Path(tempfile.mkdtemp(prefix="previous_text_reviews_", dir=output_dir.parent))
            output_dir.rename(backup / "text_reviews")
        try:
            stage.rename(output_dir)
        except OSError:
            if backup is not None:
                (backup / "text_reviews").rename(output_dir)
            raise

    write_csv(
        reports_dir / "html_text_match.csv", MAP_FIELDS,
        [{field: row[field] for field in MAP_FIELDS} for row in kept],
    )
    write_csv(reports_dir / "discarded_html.csv", MAP_FIELDS + ["discard_reason"], discarded)
    (reports_dir / "missing_descriptions.txt").write_text(
        "".join(row["text_file_name"] + "\n" for row in kept if not row["description"]),
        encoding="utf-8",
    )
    (reports_dir / "skipped_htmls.txt").write_text("".join(skipped), encoding="utf-8")
    return len(kept), len(discarded), backup


In [32]:
converted, discarded, backup = write_outputs(
    records, skipped, TEXT_OUTPUT_DIR, EXTRACTION_DIR
)
print(f"Wrote {converted:,} text files.")
print(f"Discarded {discarded:,} duplicate HTML files; skipped {len(skipped):,} pages.")
if backup is not None:
    print("Previous text files archived in:", backup)
print("Manifest and extraction reports:", EXTRACTION_DIR)


Wrote 21,312 text files.
Discarded 4 duplicate HTML files; skipped 5 pages.
Previous text files archived in: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/previous_text_reviews_wrjrsymx
Manifest and extraction reports: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/extracting_text


### 2.5 Build the review metadata CSV

Read identity fields from the manifest, never from the filename. Exclude the
source-ID text filenames in `missing_descriptions.txt`. Split each remaining
text file at the first blank line: the first paragraph is the description and
all subsequent paragraphs form the review.

All input rows are validated before replacing `review_text.csv`.


In [33]:
def review_row(path, metadata):
    text = path.read_text(encoding="utf-8-sig").strip()
    paragraphs = re.split(r"\n[ \t]*\n", text, maxsplit=1)
    if len(paragraphs) != 2 or not all(part.strip() for part in paragraphs):
        raise ValueError("expected a description followed by review paragraphs")
    return {
        **{field: metadata[field] for field in IDENTITY_FIELDS},
        "description": paragraphs[0].strip(), "review": paragraphs[1].strip(),
    }


In [34]:
if not TEXT_OUTPUT_DIR.is_dir():
    raise FileNotFoundError(f"Text directory does not exist: {TEXT_OUTPUT_DIR}")
_, manifest = read_csv(EXTRACTION_DIR / "html_text_match.csv", MAP_FIELDS)
excluded = set(
    (EXTRACTION_DIR / "missing_descriptions.txt").read_text(encoding="utf-8-sig").splitlines()
)
csv_rows, seen = [], set()
missing_description_count = 0
for metadata in manifest:
    source_id = metadata["source_id"]
    name = metadata["text_file_name"]
    if not re.fullmatch(r"[0-9a-f]{32}", source_id) or name != source_id + ".txt":
        raise ValueError(f"Invalid source ID/text filename in manifest: {name}")
    if source_id in seen:
        raise ValueError(f"Duplicate source ID in manifest: {source_id}")
    seen.add(source_id)
    if name in excluded:
        missing_description_count += 1
        continue
    csv_rows.append(review_row(TEXT_OUTPUT_DIR / name, metadata))

assert len(manifest) == len(csv_rows) + missing_description_count
assert len({row["source_id"] for row in csv_rows}) == len(csv_rows)
print(f"Prepared {len(csv_rows):,} CSV rows; excluded {missing_description_count:,} without descriptions.")


Prepared 21,271 CSV rows; excluded 41 without descriptions.


In [35]:
write_csv(REVIEW_TEXT_CSV, REVIEW_FIELDS, csv_rows)
extraction_summary = {
    "html_files": len(files),
    "text_files": converted,
    "duplicate_html_files_discarded": discarded,
    "html_files_skipped": len(skipped),
    "missing_descriptions": missing_description_count,
    "review_csv_rows": len(csv_rows),
}
pprint(extraction_summary)
print("Saved:", REVIEW_TEXT_CSV)


{'duplicate_html_files_discarded': 4,
 'html_files': 21321,
 'html_files_skipped': 5,
 'missing_descriptions': 41,
 'review_csv_rows': 21271,
 'text_files': 21312}
Saved: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/review_text.csv


## 3. Merge cleaned metadata and review text

The preceding stages have now created both CSVs and
`outputs/extracting_text/discarded_html.csv`. This stage reads those files,
matches normalized artist, album, and publication date, and accepts a link
only when there is a single candidate with a consistent author and score.
Each accepted `source_id` can appear only once.

The shared helpers from stage 2 normalize Unicode, quotes, whitespace, and
case for comparison. They preserve punctuation, and numeric scores such as
`8` and `8.0` compare equally. The output retains cleaned metadata values;
no fuzzy matching or title-repair guesses are applied.


### 3.1 Define merge fields and exclusion rules

The required match fields are artist, album, date, author, and score.
Retain the original pipeline's explicit exclusion of Prince's December 2,
2019 review of `1999`; the April 30, 2016 review remains eligible. Duplicate
metadata and HTML groups are identified below. Missing or conflicting
authors and invalid or conflicting scores prevent an accepted match.


In [36]:
MATCH_FIELDS = ["artist", "album", "reviewdate", "reviewauthor", "score"]
REQUESTED_EXCLUSIONS = {("prince", "1999", "2019-12-02")}


In [37]:
def consistency_issues(data_row, text_row):
    issues = []
    data_author = normalize_name(data_row.get("reviewauthor"))
    text_author = normalize_name(text_row.get("reviewauthor"))
    if not data_author or not text_author:
        issues.append("missing_author")
    elif data_author != text_author:
        issues.append("author_mismatch")
    try:
        if review_score(data_row.get("score")) != review_score(text_row.get("score")):
            issues.append("score_mismatch")
    except ValueError:
        issues.append("invalid_score")
    return ";".join(issues)


### 3.2 Load inputs and identify exclusion groups

Discard **every** metadata row in a repeated normalized artist/album/date
group, along with corresponding review-text rows. Also exclude the duplicate
HTML cases listed by extraction. Prince's December 2, 2019 review of `1999` is
explicitly excluded; the April 30, 2016 review remains eligible.


In [38]:
data_path = DATA_CLEAN_CSV
text_path = REVIEW_TEXT_CSV
output_dir = OUTPUTS_DIR
discarded_html_path = DISCARDED_HTML_CSV

data_fields, data_rows = read_csv(data_path, MATCH_FIELDS)

text_fields, text_rows = read_csv(text_path, REVIEW_FIELDS)

if {"source_id", "description", "review"} & set(data_fields):
    raise ValueError("cleaned metadata already contains output-only review columns")

repeated_data_keys = duplicate_keys(data_rows)

discarded_html_keys = set()

if discarded_html_path is not None:
    _, html_rows = read_csv(discarded_html_path, MATCH_FIELDS)
    discarded_html_keys = {match_key(row) for row in html_rows}

print(f"Loaded {len(data_rows):,} metadata rows and {len(text_rows):,} review-text rows.")
print(f"Found {len(repeated_data_keys):,} repeated metadata key/date groups.")


Loaded 16,811 metadata rows and 21,271 review-text rows.
Found 6 repeated metadata key/date groups.


### 3.3 Separate eligible, discarded, and invalid rows

Keep reasons for all rejected rows. Invalid match fields are unresolved.
Repeated source IDs and the requested exclusion groups are discarded.


In [39]:
data_discarded, text_discarded = [], []

data_unmatched, text_unmatched = [], []

eligible_data, eligible_text = [], []

id_counts = Counter(row["source_id"] for row in text_rows)

for rows, eligible, discarded, unmatched, is_text in (
    (data_rows, eligible_data, data_discarded, data_unmatched, False),
    (text_rows, eligible_text, text_discarded, text_unmatched, True),
):
    for row in rows:
        try:
            key = match_key(row)
        except ValueError:
            unmatched.append({**row, "merge_reason": "invalid_match_fields"})
            continue
        reason = None
        if key in repeated_data_keys:
            reason = "duplicate_metadata_key"
        elif key in REQUESTED_EXCLUSIONS:
            reason = "requested_exclusion_no_source_review"
        elif key in discarded_html_keys:
            reason = "duplicate_html_review"
        elif is_text and id_counts[row["source_id"]] > 1:
            reason = "duplicate_source_id"
        if reason:
            discarded.append({**row, "discard_reason": reason})
        else:
            eligible.append((key, row))

print(f"Eligible metadata rows: {len(eligible_data):,}; eligible text rows: {len(eligible_text):,}")
print(f"Discarded metadata rows: {len(data_discarded):,}; discarded text rows: {len(text_discarded):,}")


Eligible metadata rows: 16,797; eligible text rows: 21,265
Discarded metadata rows: 14; discarded text rows: 6


### 3.4 Find unique candidates and validate matches

Index text rows by normalized artist, album, and date. Multiple candidates,
date mismatches, author mismatches, and score mismatches stay unresolved; no
fallback joins on artist and album alone are used.


In [40]:
text_by_key = defaultdict(list)

for key, row in eligible_text:
    text_by_key[key].append(row)

text_pairs = {key[:2] for key, _ in eligible_text}

data_pairs = {key[:2] for key, _ in eligible_data}

merged = []

matched_ids = set()

failed_keys = {}


In [41]:
for key, data_row in eligible_data:
    candidates = text_by_key.get(key, [])
    if not candidates:
        reason = "review_date_mismatch" if key[:2] in text_pairs else "no_matching_review"
    elif len(candidates) != 1:
        reason = "ambiguous_review_match"
    else:
        text_row = candidates[0]
        reason = consistency_issues(data_row, text_row)
        if not reason:
            source_id = text_row["source_id"]
            if source_id in matched_ids:
                raise ValueError(f"source ID would be linked more than once: {source_id}")
            matched_ids.add(source_id)
            merged.append({
                **data_row, "source_id": source_id,
                "description": text_row["description"], "review": text_row["review"],
            })
            continue
    failed_keys[key] = reason
    data_unmatched.append({**data_row, "merge_reason": reason})

for key, text_row in eligible_text:
    if text_row["source_id"] in matched_ids:
        continue
    reason = failed_keys.get(key)
    if not reason:
        reason = "review_date_mismatch" if key[:2] in data_pairs else "no_matching_metadata"
    text_unmatched.append({**text_row, "merge_reason": reason})

assert len({row["source_id"] for row in merged}) == len(merged)
print(f"Accepted {len(merged):,} unique review links.")


Accepted 16,222 unique review links.


### 3.5 Save the merged CSV and audit reports

`full_review.csv` retains the cleaned metadata columns and adds `source_id`,
`description`, and `review`. Both unresolved CSVs retain their input columns
and add `merge_reason`. Both discarded CSVs add `discard_reason`.


In [42]:
write_csv(FULL_REVIEW_CSV, data_fields + ["source_id", "description", "review"], merged)

for filename, fields, rows in (
    ("data_clean_unmatched.csv", data_fields, data_unmatched),
    ("review_text_unmatched.csv", text_fields, text_unmatched),
):
    write_csv(output_dir / "missed_on_merge" / filename, fields + ["merge_reason"], rows)

for filename, fields, rows in (
    ("data_clean_discarded.csv", data_fields, data_discarded),
    ("review_text_discarded.csv", text_fields, text_discarded),
):
    write_csv(output_dir / "discarded_on_merge" / filename, fields + ["discard_reason"], rows)

assert len(data_rows) == len(merged) + len(data_unmatched) + len(data_discarded)

assert len(text_rows) == len(merged) + len(text_unmatched) + len(text_discarded)


In [43]:
stats = {
        "merged": len(merged),
        "data_unmatched": len(data_unmatched), "text_unmatched": len(text_unmatched),
        "data_discarded": len(data_discarded), "text_discarded": len(text_discarded),
        "duplicate_metadata_groups": len(repeated_data_keys),
        "data_unmatched_reasons": dict(Counter(row["merge_reason"] for row in data_unmatched)),
        "data_discard_reasons": dict(Counter(row["discard_reason"] for row in data_discarded)),
    }

pprint(stats)
print("Merged CSV:", FULL_REVIEW_CSV)
print("Unresolved rows:", OUTPUTS_DIR / "missed_on_merge")
print("Discarded rows:", OUTPUTS_DIR / "discarded_on_merge")


{'data_discard_reasons': {'duplicate_html_review': 2,
                          'duplicate_metadata_key': 12},
 'data_discarded': 14,
 'data_unmatched': 575,
 'data_unmatched_reasons': {'no_matching_review': 575},
 'duplicate_metadata_groups': 6,
 'merged': 16222,
 'text_discarded': 6,
 'text_unmatched': 5043}
Merged CSV: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/full_review.csv
Unresolved rows: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/missed_on_merge
Discarded rows: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/discarded_on_merge


### 3.6 Inspect accepted review links

The original metadata values are retained. `source_id` links each accepted
row to `review_text.csv`, the extraction manifest, and its text file.


In [44]:
for row in merged[:5]:
    pprint({field: row[field] for field in [
        "source_id", "artist", "album", "reviewdate", "reviewauthor", "score"
    ]})


{'album': 'Studio 1',
 'artist': 'Studio 1',
 'reviewauthor': 'Andy Battaglia',
 'reviewdate': '2009-02-18',
 'score': '8.5',
 'source_id': '37ab31b68172e82813f2a369f582cbaa'}
{'album': 'The Great Santa Barbara Oil Slick',
 'artist': 'John Fahey',
 'reviewauthor': 'Mark Richardson',
 'reviewdate': '2005-02-13',
 'score': '8.2',
 'source_id': '262b7d646d70d63ae9dda3963574311e'}
{'album': 'Too Much Guitar',
 'artist': 'Reigning Sound',
 'reviewauthor': 'Stephen M. Deusner',
 'reviewdate': '2004-08-19',
 'score': '8.3',
 'source_id': '63d7991eedbf90992723439b6b87ff0b'}
{'album': 'After the Last',
 'artist': 'The Red Thread',
 'reviewauthor': 'Chris Dahlen',
 'reviewdate': '2003-07-17',
 'score': '7.3',
 'source_id': 'ece0303b76fc865d6d8429b558fd8680'}
{'album': 'With Blasphemy So Heartfelt',
 'artist': 'Jessica Lea Mayfield',
 'reviewauthor': 'Stephen M. Deusner',
 'reviewdate': '2008-11-11',
 'score': '8.2',
 'source_id': 'e7a2b5ec09d43c300f2ba9381e1b06fe'}


## 4. Check the completed workflow and find its outputs

The final dataset contains all columns from `data_clean.csv` plus
`source_id`, `description`, and `review`. It is the **matched subset** of the
cleaned metadata, so its row count can be smaller. `source_id` connects a row
to the review-text CSV, extraction manifest, and text file. `source_row`
connects it back to the raw metadata; `review_id` identifies it in the cleaned
export.

| Output path (relative to `OUTPUTS_DIR`) | Purpose |
| --- | --- |
| `data_clean.csv` | Cleaned metadata, audio features, helper fields, and quality flags |
| `data_cleaning/log_df.csv` | Cleaning actions and affected counts |
| `review_text.csv` | Source ID, artist, album, date, author, score, description, and review |
| `text_reviews/<source_id>.txt` | Description and review paragraphs for each retained HTML review |
| `extracting_text/html_text_match.csv` | Mapping between source IDs, HTML filenames, text filenames, and identity fields |
| `extracting_text/discarded_html.csv` | Duplicate HTML groups and their exclusion reason |
| `extracting_text/missing_descriptions.txt` | Retained text files omitted from `review_text.csv` because their descriptions are missing |
| `extracting_text/skipped_htmls.txt` | HTML inputs that could not be parsed, with reasons |
| `full_review.csv` | Validated metadata/text links for analysis |
| `missed_on_merge/data_clean_unmatched.csv` | Unresolved metadata rows with `merge_reason` |
| `missed_on_merge/review_text_unmatched.csv` | Unresolved review-text rows with `merge_reason` |
| `discarded_on_merge/data_clean_discarded.csv` | Excluded metadata rows with `discard_reason` |
| `discarded_on_merge/review_text_discarded.csv` | Excluded review-text rows with `discard_reason` |

The checks below account for every metadata and text row as matched,
unresolved, or discarded, confirm source-ID uniqueness, and verify that the
three main output files exist. Review the reason counts printed above to
understand any exclusions before proceeding to analysis.


In [45]:
assert len(data_rows) == len(merged) + len(data_unmatched) + len(data_discarded)
assert len(text_rows) == len(merged) + len(text_unmatched) + len(text_discarded)
assert len(matched_ids) == len(merged)
assert all(path.is_file() for path in (DATA_CLEAN_CSV, REVIEW_TEXT_CSV, FULL_REVIEW_CSV))

workflow_summary = pd.DataFrame([
    {"stage": "Raw metadata", "rows": ROWS_AT_START, "file": str(RAW_DATA_CSV)},
    {"stage": "Cleaned metadata", "rows": len(data_rows), "file": str(DATA_CLEAN_CSV)},
    {"stage": "Review text", "rows": len(text_rows), "file": str(REVIEW_TEXT_CSV)},
    {"stage": "Merged reviews", "rows": len(merged), "file": str(FULL_REVIEW_CSV)},
])
display(workflow_summary)
print("Workflow complete. Final dataset:", FULL_REVIEW_CSV)


,stage,rows,file
0,Raw metadata,18403,/Users/jaypark/Classwork/COS424/objectively-re...
1,Cleaned metadata,16811,/Users/jaypark/Classwork/COS424/objectively-re...
2,Review text,21271,/Users/jaypark/Classwork/COS424/objectively-re...
3,Merged reviews,16222,/Users/jaypark/Classwork/COS424/objectively-re...


Workflow complete. Final dataset: /Users/jaypark/Classwork/COS424/objectively-reviewed-cos424/outputs/full_review.csv


## 5. NLP baseline: review language and outcomes

The merge stage above writes `outputs/full_review.csv`: this contains the matched metadata, Pitchfork score, Spotify audio features, and review text. The following analysis uses TF-IDF unigram/bigram features and Ridge regression to predict the score and nine audio features. It uses a reproducible 80/20 held-out split and compares against a training-mean baseline. The vocabulary and target scaling are fit on training rows only. This is a simple language-modeling baseline; it does not replace the proposed word-by-word embedding analysis.

In [46]:
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from preliminary_results import fit_tfidf_baseline

nlp_data = pd.read_csv(FULL_REVIEW_CSV)
metrics, top_terms, model_info = fit_tfidf_baseline(nlp_data)
print(
    f"Complete reviews: {model_info['complete_rows']:,}; "
    f"training: {model_info['train_rows']:,}; test: {model_info['test_rows']:,}; "
    f"training vocabulary: {model_info['vocabulary_size']:,} unigrams/bigrams"
)

metric_table = metrics.pivot(
    index="target", columns="model", values=["r2", "mae", "rmse"]
)
display(metric_table.round(3))

print("Largest positive/negative TF-IDF coefficients for selected outcomes:")
selected_terms = top_terms[
    top_terms["target"].isin(
        ["score", "danceability", "energy", "acousticness", "valence"]
    )
].groupby(["target", "direction"], sort=False).head(5)
display(selected_terms.reset_index(drop=True).round({"coefficient": 3}))


Complete reviews: 16,213; training: 12,970; test: 3,243; training vocabulary: 30,000 unigrams/bigrams


r2                                   mae                                  rmse                       
model            TF-IDF + Ridge Training-mean baseline TF-IDF + Ridge Training-mean baseline TF-IDF + Ridge Training-mean baseline
target                                                                                                                            
acousticness              0.270                 -0.000          0.177                  0.213          0.221                  0.258
danceability              0.290                 -0.000          0.106                  0.130          0.135                  0.160
energy                    0.263                 -0.000          0.135                  0.162          0.172                  0.200
instrumentalness          0.295                 -0.002          0.210                  0.263          0.256                  0.305
liveness                  0.085                 -0.001          0.064                  0.067          0.096                  0.100
loudness                  0.224                 -0.000          2.667                  3.129          3.717                  4.220
score                     0.311                 -0.001          0.758                  0.937          1.043                  1.256
speechiness               0.293                 -0.000          0.046                  0.059          0.073                  0.086
tempo                     0.082                 -0.001         10.683                 11.252         14.271                 14.907
valence                   0.244                 -0.000          0.129                  0.150          0.158                  0.182

Largest positive/negative TF-IDF coefficients for selected outcomes:


,target,direction,term,coefficient
0,score,positive,reissue,0.734
1,score,positive,also,0.674
2,score,positive,jazz,0.584
3,score,positive,perfect,0.565
4,score,positive,each,0.556
5,score,negative,unfortunately,-1.208
6,score,negative,problem,-1.151
7,score,negative,least,-1.135
8,score,negative,at least,-1.023
9,score,negative,worse,-0.969


### Interpreting the NLP baseline

Held-out R² indicates whether this bag-of-words model improves on predicting the training mean. TF-IDF ignores word order and context. Ridge coefficients show associations conditional on other correlated terms; they are not causal effects or true word embeddings. Since reviews can share authors or artists across a random split, grouped evaluation by reviewer, artist, or time is a useful next step. Any learned representation or other data-derived transformation must be fit on training folds only to avoid leakage.